# Day 5 Sample Notebook: Treatment Pathways

**OHDSI / OMOP Train-the-Trainer, ALS Therapy Development Institute**

Companion to the Day 5 module. You will reconstruct each patient's ordered sequence of ALS therapies
and summarize the dominant first-line treatment and the most common pathways.

> **Site specific.** This notebook builds a synthetic CDM in memory so it runs in Colab with no credentials and no PHI. To use real data, replace the SQLite connection with your site's governed CDM connection (Databricks, Snowflake, Postgres, BigQuery, and so on). The SQL logic is the same; only the connection changes. Not everyone uses Databricks.

### Objectives
1. Map drug products to ingredients and order each person's exposures over time.
2. Build per-person treatment sequences.
3. Summarize first-line treatment and the top pathways.
4. Interpret what a pathway analysis can and cannot tell you.

## Step 0: Build the synthetic CDM

In [ ]:
import sqlite3, numpy as np, pandas as pd
np.random.seed(7)

def build_als_cdm(n=250):
    """Build a tiny SYNTHETIC ALS-flavored OMOP CDM in memory. No real data, no PHI."""
    con = sqlite3.connect(":memory:"); cur = con.cursor()
    cur.executescript("""
    CREATE TABLE concept(concept_id INT, concept_name TEXT, domain_id TEXT, vocabulary_id TEXT,
        concept_class_id TEXT, standard_concept TEXT, concept_code TEXT);
    CREATE TABLE concept_ancestor(ancestor_concept_id INT, descendant_concept_id INT, min_levels_of_separation INT);
    CREATE TABLE person(person_id INT, gender_concept_id INT, year_of_birth INT);
    CREATE TABLE observation_period(person_id INT, observation_period_start_date TEXT, observation_period_end_date TEXT);
    CREATE TABLE condition_occurrence(person_id INT, condition_concept_id INT, condition_start_date TEXT);
    CREATE TABLE drug_exposure(person_id INT, drug_concept_id INT, drug_exposure_start_date TEXT);
    CREATE TABLE measurement(person_id INT, measurement_concept_id INT, measurement_date TEXT, value_as_number REAL);
    CREATE TABLE procedure_occurrence(person_id INT, procedure_concept_id INT, procedure_date TEXT);
    """)
    concepts=[
     (35748069,"Amyotrophic lateral sclerosis","Condition","SNOMED","Clinical Finding","S","86044005"),
     (4134454,"Bulbar onset","Observation","SNOMED","Clinical Finding","S","230258005"),
     (1314865,"riluzole","Drug","RxNorm","Ingredient","S","9468"),
     (19077572,"riluzole 50 MG Oral Tablet","Drug","RxNorm","Clinical Drug","S","316158"),
     (1326727,"edaravone","Drug","RxNorm","Ingredient","S","1546020"),
     (40234834,"edaravone 30 MG/100mL","Drug","RxNorm","Clinical Drug","S","1546022"),
     (9990001,"sodium phenylbutyrate / taurursodiol","Drug","RxNorm","Ingredient","S","2630918"),
     (9990002,"tofersen","Drug","RxNorm","Ingredient","S","2629000"),
     (1304643,"baclofen","Drug","RxNorm","Ingredient","S","1292"),
     (4052536,"Gastrostomy","Procedure","SNOMED","Procedure","S","36245001"),
     (3024171,"Forced vital capacity","Measurement","LOINC","Clinical Obs","S","19868-9"),
     (8532,"FEMALE","Gender","Gender","Gender","S","F"),(8507,"MALE","Gender","Gender","Gender","S","M"),
     (0,"No matching concept","Metadata","None","Undefined",None,"0")]
    cur.executemany("INSERT INTO concept VALUES(?,?,?,?,?,?,?)",concepts)
    cur.executemany("INSERT INTO concept_ancestor VALUES(?,?,?)",[
     (1314865,19077572,1),(1314865,1314865,0),(1326727,40234834,1),(1326727,1326727,0),
     (9990001,9990001,0),(9990002,9990002,0),(1304643,1304643,0)])
    persons=[];obsp=[];cond=[];drug=[];meas=[];proc=[]
    for pid in range(1,n+1):
        sex=int(np.random.choice([8532,8507])); yob=int(np.random.normal(1958,11))
        persons.append((pid,sex,yob))
        dx_year=np.random.randint(2018,2024); dx=f"{dx_year}-{np.random.randint(1,13):02d}-15"
        cond.append((pid,35748069,dx))
        bulbar=np.random.rand()<0.3
        if bulbar: cond.append((pid,4134454,dx))
        prior=int(np.random.choice([400,500,800,200,100],p=[.4,.2,.2,.1,.1]))
        start=pd.Timestamp(dx)-pd.Timedelta(days=prior)
        end=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(365,1500)))
        obsp.append((pid,start.date().isoformat(),end.date().isoformat()))
        base_fvc=float(np.clip(np.random.normal(85-(10 if bulbar else 0),12),30,110))
        r=np.random.rand()
        if r<0.08: pass                                  # missing (completeness)
        elif r<0.11: meas.append((pid,3024171,dx,float(np.random.choice([0,250.0]))))  # implausible (plausibility)
        else: meas.append((pid,3024171,dx,round(base_fvc,1)))
        idx=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(0,40)))
        if np.random.rand()<0.85:
            drug.append((pid,19077572,idx.date().isoformat()))
            if np.random.rand()<0.45:
                drug.append((pid,40234834,(idx+pd.Timedelta(days=int(np.random.randint(60,300)))).date().isoformat()))
        else:
            drug.append((pid,40234834,idx.date().isoformat()))
        if np.random.rand()<0.15:
            drug.append((pid,9990001,(idx+pd.Timedelta(days=int(np.random.randint(100,500)))).date().isoformat()))
        if bulbar and np.random.rand()<0.4: drug.append((pid,1304643,idx.date().isoformat()))
        if np.random.rand()<0.05: drug.append((pid,0,idx.date().isoformat()))   # unmapped (conformance)
        age=dx_year-yob
        logit=-2.0+0.04*(age-60)+1.1*bulbar-0.03*(base_fvc-80)
        if np.random.rand()<1/(1+np.exp(-logit)):
            proc.append((pid,4052536,(pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(60,360)))).date().isoformat()))
    cur.executemany("INSERT INTO person VALUES(?,?,?)",persons)
    cur.executemany("INSERT INTO observation_period VALUES(?,?,?)",obsp)
    cur.executemany("INSERT INTO condition_occurrence VALUES(?,?,?)",cond)
    cur.executemany("INSERT INTO drug_exposure VALUES(?,?,?)",drug)
    cur.executemany("INSERT INTO measurement VALUES(?,?,?,?)",meas)
    cur.executemany("INSERT INTO procedure_occurrence VALUES(?,?,?)",proc)
    con.commit(); return con

con = build_als_cdm()
def q(sql): return pd.read_sql_query(sql, con)
print("Synthetic ALS CDM ready:", q("SELECT COUNT(*) n FROM person").n[0], "persons")

## Step 1: Ingredient-level exposures, ordered in time
Pathway analysis works at the ingredient level. We roll products up to their ingredient and sort by date.

In [ ]:
ex = q("""
SELECT de.person_id, ing.concept_name AS ingredient, de.drug_exposure_start_date AS dt
FROM drug_exposure de
JOIN concept_ancestor ca ON ca.descendant_concept_id = de.drug_concept_id
JOIN concept ing ON ing.concept_id = ca.ancestor_concept_id
WHERE ing.concept_class_id = 'Ingredient'
  AND ing.concept_name IN ('riluzole','edaravone','sodium phenylbutyrate / taurursodiol','tofersen')
ORDER BY de.person_id, de.drug_exposure_start_date
""")
print("ingredient-level exposure rows:", len(ex)); ex.head()

## Step 2: Build each person's pathway
Collapse repeats and keep first appearance order, so "riluzole then edaravone" is one pathway.

In [ ]:
def seq(g):
    out=[]
    for name in g.sort_values("dt").ingredient:
        if name not in out: out.append(name)
    return " -> ".join(out)

paths = ex.groupby("person_id").apply(seq).rename("pathway").reset_index()
print("people with a pathway:", len(paths))
paths.head()

## Step 3: First-line and top pathways

In [ ]:
first_line = ex.sort_values(["person_id","dt"]).groupby("person_id").first()["ingredient"]
print("First-line treatment distribution:")
print(first_line.value_counts(), "\n")
print("Top treatment pathways:")
print(paths.pathway.value_counts().head(8))

## Step 4: Visualize the top pathways (optional)

In [ ]:
import matplotlib.pyplot as plt
top = paths.pathway.value_counts().head(6)[::-1]
plt.figure(figsize=(8,3.5))
plt.barh(top.index, top.values, color="#904199")
plt.xlabel("patients"); plt.title("Most common ALS treatment pathways (synthetic)")
plt.tight_layout(); plt.show()

## Interpretation
A pathway analysis is **descriptive**. It shows what sequences occurred, not whether one therapy
caused a better outcome. Selection effects are everywhere: sicker patients may escalate sooner. Treat
pathways as a map of practice patterns and a hypothesis generator, not as causal evidence.

## Your turn
1. What share of patients received only a single therapy (no arrow in their pathway)?
2. Recompute first-line treatment for bulbar-onset patients only. Does it differ?
3. Add baclofen to the ingredient filter. Why might including a symptomatic (non-disease-modifying) drug distort a pathway analysis?

<details><summary>Answer key</summary>

1. `(~paths.pathway.str.contains(" -> ")).mean()` gives the single-therapy share.
2. Join `paths`/`first_line` to the bulbar flag from `condition_occurrence` (concept 4134454) and recompute.
3. Symptomatic drugs are not part of the disease-modifying sequence, so including them mixes two
   different clinical questions and inflates apparent pathway diversity. Define the drug set deliberately.
</details>